In [2]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import SpanSelector
from tkinter import filedialog, Tk

In [3]:
import plotly.express as px
colorscales = px.colors.named_colorscales()
print(colorscales)

['aggrnyl', 'agsunset', 'blackbody', 'bluered', 'blues', 'blugrn', 'bluyl', 'brwnyl', 'bugn', 'bupu', 'burg', 'burgyl', 'cividis', 'darkmint', 'electric', 'emrld', 'gnbu', 'greens', 'greys', 'hot', 'inferno', 'jet', 'magenta', 'magma', 'mint', 'orrd', 'oranges', 'oryel', 'peach', 'pinkyl', 'plasma', 'plotly3', 'pubu', 'pubugn', 'purd', 'purp', 'purples', 'purpor', 'rainbow', 'rdbu', 'rdpu', 'redor', 'reds', 'sunset', 'sunsetdark', 'teal', 'tealgrn', 'turbo', 'viridis', 'ylgn', 'ylgnbu', 'ylorbr', 'ylorrd', 'algae', 'amp', 'deep', 'dense', 'gray', 'haline', 'ice', 'matter', 'solar', 'speed', 'tempo', 'thermal', 'turbid', 'armyrose', 'brbg', 'earth', 'fall', 'geyser', 'prgn', 'piyg', 'picnic', 'portland', 'puor', 'rdgy', 'rdylbu', 'rdylgn', 'spectral', 'tealrose', 'temps', 'tropic', 'balance', 'curl', 'delta', 'oxy', 'edge', 'hsv', 'icefire', 'phase', 'twilight', 'mrybm', 'mygbm']


In [1]:
import napari
viewer = napari.Viewer()    

/Users/maximiliansenftleben/miniconda3/envs/7904/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
# ask for folder name
root = Tk()
root.withdraw()
data_path = filedialog.askdirectory(title="Select MINFLUX data folder")
print(f"Selected data path: {data_path}")
folders = [entry.name for entry in os.scandir(data_path) if entry.is_dir()]

Selected data path: /Users/maximiliansenftleben/Data/8376_alm/8376_minflux/data/multicolor


## Functions (Will be saved as .py file once done)

In [3]:
# functions
def save_to_csv(df, save_path):
    df.to_csv(save_path)

def np_to_df(np_data):
    df = pd.DataFrame(np_data.tolist(), columns=np_data.dtype.names)

    column_to_split = ["loc", "lnc", "dcr"]

    vals = ['x', 'y', 'z'] 
    for column in column_to_split:

        split_data = pd.DataFrame(df[column].tolist(), columns=[f"{column}_{vals[i]}" for i in range(len(df[column][0]))])
        df = pd.concat([df.drop(column, axis=1), split_data], axis=1)
    return df

%matplotlib widget
def efo_filtering_box(MFX_Data, save_path, to_save):

    MFX_EFO = MFX_Data['efo'][MFX_Data['itr'] == np.max(MFX_Data['itr'])]
    bin_size = 3e3
    bins = np.arange(MFX_EFO.min(), MFX_EFO.max() + bin_size, bin_size)
    counts, _ = np.histogram(MFX_EFO, bins=bins)

    fig, ax = plt.subplots()
    fig.canvas.header_visible = False
    fig.canvas.footer_visible = False

    ax.hist(MFX_EFO, bins=bins, edgecolor='darkorange', color='darkorange')
    
    def onselect(xmin, xmax):
        to_save[save_path] = np_to_df(MFX_Data[(MFX_Data['efo'] >= xmin) & (MFX_Data['efo'] <= xmax)])

    span = SpanSelector(
        ax, onselect,
        direction='horizontal',
        useblit=True,
        props=dict(facecolor='yellow', alpha=0.3, edgecolor='black', linewidth=5),
        button=1,
        interactive=True,
        drag_from_anywhere=True
    )

    fig._span = span
    
    # set default span to min and max of EFO values
    ax.set_xlim(MFX_EFO.min() - 50000, MFX_EFO.max() + 50000)
    span.extents = (MFX_EFO.min(), MFX_EFO.max())
    onselect(MFX_EFO.min(), MFX_EFO.max())

    ax.set_xlabel('EFO')
    ax.set_ylabel('Count')
    ax.set_title(os.path.basename(save_path).split('.')[0])
    return fig, ax, span

In [4]:
min_trace_length = 3
z_correction_factor = 0.7
scale_factor = 1000000000 # what unit 
save_folder = data_path + "_filtered"
os.makedirs(save_folder, exist_ok=True)

to_save = {}

# find all npy files in the data_path and its subfolders
npy_files = glob.glob(os.path.join(data_path, "**", "*.npy"), recursive=True)
stats = {}
for file in npy_files:
    print(file)
    try:
        save_path = os.path.join(save_folder, f'{os.path.splitext(os.path.basename(file))[0]}.csv')
        MFX_Data = np.load(file)
        
        # total imaging time
        total_tim =  MFX_Data["tim"][-1] - MFX_Data["tim"][0]
        stats[save_path] = {}
        stats[save_path][f'Total imaging time (min): '] = f'{total_tim/60:.2f}'
        stats[save_path][f'Total localizations: '] = f'{len(MFX_Data)}'
        # keep valid localizations only (vld field)
        MFX_Data =  MFX_Data[MFX_Data['vld'] == True]
        stats[save_path][f'Valid localizations: '] = f'{len(MFX_Data)}'
        print(len(MFX_Data))
        # z correction
        MFX_Data['loc'][:,-1] = MFX_Data['loc'][:,-1]*z_correction_factor
        MFX_Data['loc'] = MFX_Data['loc']*scale_factor
        
        # take last iteration only
        MFX_Data_vld_fnl =  MFX_Data[(MFX_Data['itr'] == max(MFX_Data['itr'])) & (MFX_Data['itr'] == max(MFX_Data['itr']))]
        MFX_Data_vld_fnl = MFX_Data.copy()  # to avoid warning
        stats[save_path][f'Final iteration localizations: '] = f'{len(MFX_Data_vld_fnl)}'

        # trace filtering
        unique_tids, inv_idx, locs_per_tid = np.unique(MFX_Data_vld_fnl['tid'],  return_inverse=True,return_counts=True) 
        MFX_Data_vld_fnl_filt = MFX_Data_vld_fnl[locs_per_tid[inv_idx]>=5] 
        print(len(MFX_Data_vld_fnl_filt))
        stats[save_path][f'Localizations after trace filtering: '] = f'{len(MFX_Data_vld_fnl_filt)}'

        # efo thresholding
        '''
        fig, ax, span = efo_filtering_box(MFX_Data_vld_fnl_filt, save_path, to_save)
        xmin, xmax = span.extents

        # TODO might want to make this later to gain speed
        MFX_data_filtered = MFX_Data_vld_fnl_filt[(MFX_Data_vld_fnl_filt['efo'] >= xmin) & (MFX_Data_vld_fnl_filt['efo'] <= xmax)]
        df = np_to_df(MFX_data_filtered)
        to_save[save_path] = df
        
        plt.show()
        '''
    except FileNotFoundError:
        print(f"No .npy files found in {data_path}.")
        continue

print(stats)

NameError: name 'data_path' is not defined

### Do median calculation (using the LOC values)

In the exported data there are two types of localization coordinates: the MINFLUX Beamline Monitoring (MBM) corrected, called LOC, and the raw localizations, named LNC which stands for Localizations not corrected. Both fields have three columns, one for each 3D cartesian coordinate: x, y, and z.

In [5]:
for save_path, df in to_save.items():
    num_after_efo = len(df)
    stats[save_path][f'Localizations after EFO filtering: '] = f'{num_after_efo}'
    stats[save_path][f'Percentage remaining localizations: '] = f'% {(num_after_efo / int(stats[save_path][f"Total localizations: "]) * 100):.2f}'
    print(f'_________________________{save_path.split("/")[-1]}_______________________________________________________')
    stats_df = pd.DataFrame.from_dict(stats[save_path], orient='index', columns=['Value'])
    print(stats_df)

    dims = [i.replace('loc_', '') for i in df.columns.to_list() if i.startswith('loc_')] # automatic detection of dimensions
    index = os.path.basename(save_path).split('.')[0]
    avg_df = pd.DataFrame()
    for dim in dims:
        avg_df[f'loc_{dim}_mean'] = df.groupby('tid')[f'loc_{dim}'].mean()     # TODO median or mean?
        avg_df[f'loc_{dim}_std'] = df.groupby('tid')[f'loc_{dim}'].std()          # std
    
    avg_df['n'] = df.groupby('tid')[f'loc_{dims[0]}'].count()
    avg_df['tim_tot'] = df.groupby('tid')[f'tim'].sum()

    # save as csv
    avg_save_path = os.path.join(save_folder, f'{index}_stats.csv')    
    print(f"Saving filtered data to:   {save_path}")
    save_to_csv(df, save_path)      # save filtered data
    print(f"Saving stats to:           {avg_save_path}")
    avg_df.to_csv(avg_save_path)

_________________________250603-133132_R2_minflux.csv_______________________________________________________
                                         Value
Total imaging time (min):                43.21
Total localizations:                    179237
Valid localizations:                    179237
Final iteration localizations:          179237
Localizations after trace filtering:    179237
Localizations after EFO filtering:       75232
Percentage remaining localizations:    % 41.97
Saving filtered data to:   /Users/maximiliansenftleben/Data/8376_alm/8376_minflux/data/250603_BRUNO_3D_Alfa1_filtered/250603-133132_R2_minflux.csv
Saving stats to:           /Users/maximiliansenftleben/Data/8376_alm/8376_minflux/data/250603_BRUNO_3D_Alfa1_filtered/250603-133132_R2_minflux_stats.csv
_________________________250603-104741_R1_minflux.csv_______________________________________________________
                                         Value
Total imaging time (min):               150.99
Total locali

In [ ]:
# STATS
# - localization precision mean of all std per trace
# - number of photons
#   - video from maria given to bruno
# - rendering of tid and locs with napari
# - A + B or two colors align with mbm from zarr based on the gold bead localization
# - code from maria given to ana
# - downside of daniel visualization can be improved


In [7]:
import napari
viewer = napari.Viewer()

In [8]:
MFX_Data

array([( True, False,  True, False, 2,   12.2740848 ,   296,  5, 0, 0, 0, [1.23696892e+04, 2.27118413e+04, 9.17299628e-06], [1.23696756e-05, 2.27118539e-05, 0.00000000e+00], 184,  0, 36475.37 ,        nan,     0.   , 0.    , [0.3152, 0.6846]),
       ( True, False, False, False, 2,   12.29134128,   296,  5, 0, 0, 1, [1.23268275e+04, 2.26616719e+04, 1.18191063e-05], [1.23268100e-05, 2.26616883e-05, 0.00000000e+00], 164,  0, 78319.01 ,        nan,     0.   , 0.    , [0.378 , 0.622 ]),
       ( True, False, False, False, 2,   12.2940646 ,   296,  5, 0, 0, 2, [1.23125670e+04, 2.26401554e+04, 1.22375870e-05], [1.23125489e-05, 2.26401722e-05, 0.00000000e+00], 103, 48, 98376.31 , 47690.016 ,     0.   , 0.4656, [0.4304, 0.5693]),
       ...,
       ( True,  True, False, False, 1, 1342.46685265, 40720, 44, 0, 0, 4, [1.11257075e+04, 2.29277578e+04, 4.51710864e-04], [1.11429779e-05, 2.29124500e-05, 0.00000000e+00], 151, 73, 24036.932, 12088.094 , 10506.208, 0.4834, [0.4297, 0.5703]),
       ( Tru